# 05 · Evaluation
Aggregate metrics, per-class breakdown (where aggregate accuracy hides failures on rare classes), detection rate at fixed false-positive rates, and significance tests.

In [ ]:
%cd -q ..
# notebooks run from the project root so relative config paths resolve

In [ ]:
import logging
import pandas as pd
from IPython.display import Image, display

from ids import experiments as ex
from ids.config import load_config, results_dir

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(name)s: %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.max_columns", 50)

# Use configs/smoke.yaml to try a notebook on synthetic data (python -m ids.synthetic first)
cfg = load_config("configs/default.yaml")
OUT = results_dir(cfg)

In [ ]:
summary = ex.report(cfg)
summary

In [ ]:
display(Image(OUT / "model_comparison.png"))
display(Image(OUT / "roc_pr.png"))

### Per-class recall
H2: accuracy is high everywhere, but recall on the rarest classes should be visibly lower, and should move with the imbalance strategy.

In [ ]:
display(Image(OUT / "per_class_recall.png"))

In [ ]:
best = summary.iloc[0]
display(Image(OUT / "runs" / f"{best.model}__{best.strategy}__cm.png"))

### Is the difference real?
`cv_p_value`: Nadeau–Bengio corrected resampled t-test on per-fold macro-F1. `mcnemar_p_value`: McNemar's test on the shared test set. A plain paired t-test over folds is not used because it overstates significance.

In [ ]:
sig = ex.run_stats(cfg)
sig.sort_values(["strategy", "cv_p_value"])